# Train FRIDAY's own Piper voice (Google Colab)

Fine-tunes a Piper voice on recordings of the voice you want her to have, and
exports the `.onnx` + `.onnx.json` pair that `PiperTTSProvider` feeds to the
`piper` binary.

**Fine-tune, never from scratch.** From-scratch VITS wants tens of hours and
days of GPU; fine-tuning a released checkpoint wants 20-30 minutes of clean
audio and an hour or two on a T4. If you have less than ~15 minutes, stop —
the result will sound worse than the stock voice you started from.

**Runtime:** *Runtime ▸ Change runtime type ▸ GPU*.


## 1. Install

Pinned deliberately. Piper's training code moved into `piper1-gpl` upstream and
the old `piper-train` entry points come and go; unpinned, this notebook breaks
the way the wake-word one did.

In [ ]:
!pip install -q piper-tts==1.2.0 piper-phonemize-cross
!git clone -q --depth 1 -b v1.2.0 https://github.com/rhasspy/piper.git piper-src
%cd piper-src/src/python
!pip install -q -e . 2>&1 | tail -2
!pip install -q "torchmetrics==0.11.4" "pytorch-lightning==1.9.5"
%cd /content
import piper_phonemize
print('PIPER OK')

## 2. Your recordings

An LJSpeech layout: `dataset/wavs/*.wav` plus `dataset/metadata.csv` with
`id|transcript` per line, no header.

Quality beats quantity. One microphone, one room, no background music, and
transcripts that match the audio exactly — Piper learns your mistakes as
faithfully as your voice.

In [ ]:
from google.colab import files
import os, zipfile, glob
os.makedirs('dataset', exist_ok=True)
up = files.upload()          # a zip with wavs/ and metadata.csv
for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('dataset')

wavs = glob.glob('dataset/**/*.wav', recursive=True)
meta = glob.glob('dataset/**/metadata.csv', recursive=True)
assert wavs, 'no wavs found'
assert meta, 'no metadata.csv found'
lines = open(meta[0]).read().strip().splitlines()
print(len(wavs), 'clips |', len(lines), 'transcript lines')
assert abs(len(wavs) - len(lines)) <= 1, 'clips and transcripts disagree - fix before training'

## 3. Preprocess

In [ ]:
!python -m piper_train.preprocess \
    --language en-us \
    --input-dir dataset \
    --output-dir train \
    --dataset-format ljspeech \
    --single-speaker \
    --sample-rate 22050
import os; print('config:', os.path.exists('train/config.json'))

## 4. Fine-tune

`--quality medium` matches the checkpoint below. The two must agree: a medium
checkpoint into a high-quality run is a shape mismatch three screens of stack
trace long.

In [ ]:
CKPT_URL = ('https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/'
            'en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt')
!wget -q -O base.ckpt "$CKPT_URL" && ls -lh base.ckpt

!python -m piper_train \
    --dataset-dir train \
    --accelerator gpu --devices 1 \
    --batch-size 16 \
    --validation-split 0.05 \
    --num-test-examples 2 \
    --quality medium \
    --checkpoint-epochs 5 \
    --resume_from_checkpoint base.ckpt \
    --max_epochs 2000 \
    --precision 32

## 5. Export

Stop the cell above when the samples sound right (checkpoints are written as it
goes), then export the newest one.

In [ ]:
import glob, os
ckpts = sorted(glob.glob('train/lightning_logs/*/checkpoints/*.ckpt'), key=os.path.getmtime)
assert ckpts, 'no checkpoint was written - did training run at all?'
latest = ckpts[-1]
print('exporting', latest)

!python -m piper_train.export_onnx "$latest" friday.onnx
!cp train/config.json friday.onnx.json
print('EXPORTED', os.path.getsize('friday.onnx'), 'bytes')

## 6. Install her voice

Both files, same directory, names matching — the binary finds the `.json` by
appending to the model path, so renaming one and not the other fails at
synthesis time with nothing useful in the message:

```
models/voice/friday.onnx
models/voice/friday.onnx.json
```

Then point FRIDAY at it:

```bash
FRIDAY_TTS_PROVIDER=piper
FRIDAY_PIPER_MODEL=models/voice/friday.onnx
```

Check it before trusting it:

```bash
echo "All systems nominal, boss." | piper --model models/voice/friday.onnx --output_file test.wav
```

In [ ]:
from google.colab import files
files.download('friday.onnx'); files.download('friday.onnx.json')